# From linear algebra to many-body physicsExecutable companion to chapter 2 of *Quantum mechanics for Many-particleSystems*.  Everything in that chapter is linear algebra applied toantisymmetric wave functions, and every claim made there can be checkednumerically.  We do so here for:1. the single-particle basis and its one- and two-body matrix elements;2. the Slater determinant: antisymmetry, the Pauli principle, and the fact   that its occupation numbers are exactly 1 and 0;3. basis changes: $\det(\boldsymbol{C})\det(\boldsymbol{\Phi})$, and the   invariance of the energy under unitary rotations of the occupied orbitals;4. the energy functional $E[\Psi]$ and the cost of the two-body matrix   elements;5. Hartree-Fock as a repeated eigenvalue problem, and the correlation energy   that a single determinant misses.The model is spinless fermions in a one-dimensional harmonic trap with asoftened Coulomb repulsion.  Spin is suppressed: it adds bookkeeping butnothing conceptual.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.set_printoptions(precision=8, suppress=True)rng = np.random.default_rng(2026)

## 1. The single-particle basisWe use the harmonic oscillator eigenfunctions$$\phi_n(x) = \left(2^n n!\sqrt{\pi}\right)^{-1/2} H_n(x)\,e^{-x^2/2},\qquad\hat{h}_0\phi_n = \left(n+\tfrac12\right)\phi_n ,$$so the one-body matrix is diagonal, $\langle p|\hat h_0|q\rangle =\varepsilon_p\delta_{pq}$ — exactly the situation assumed in the chapter.  Thetwo-body elements$$\langle pq|v|rs\rangle = \iint \phi_p(x)\phi_q(y)\,v(x,y)\,\phi_r(x)\phi_s(y)   \,dx\,dy$$factorise into pair densities for a local interaction, which is thepair-index picture used in the low-rank factorisation of chapter 1.

In [ ]:
class HarmonicOscillatorBasis:    """Harmonic oscillator orbitals with one- and two-body matrix elements."""    def __init__(self, n_orb=8, n_grid=601, rmax=8.0, strength=1.0,                 softening=0.5):        self.n_orb = n_orb        self.strength = strength        self.softening = softening        self.x, self.h = np.linspace(-rmax, rmax, n_grid, retstep=True)        self.phi = self._hermite_functions()        self.epsilon = np.arange(n_orb) + 0.5    def _hermite_functions(self):        x = self.x        phi = np.zeros((self.n_orb, len(x)))        gauss = np.exp(-x**2 / 2.0)        H_prev, H = np.zeros_like(x), np.ones_like(x)        for n in range(self.n_orb):            # H_{n+1} = 2x H_n - 2n H_{n-1}            phi[n] = H * gauss            H_prev, H = H, 2.0 * x * H - 2.0 * n * H_prev        phi /= np.sqrt(self.h * np.sum(phi**2, axis=1))[:, None]        return phi    @property    def overlap(self):        return self.h * (self.phi @ self.phi.T)    @property    def one_body(self):        return np.diag(self.epsilon)    def two_body(self):        """v[p,q,r,s] = <pq|v|rs>."""        n, x, h = self.n_orb, self.x, self.h        K = self.strength / np.sqrt((x[:, None] - x[None, :])**2                                    + self.softening**2)        rho = (self.phi[:, None, :] * self.phi[None, :, :]).reshape(n*n, -1)        V = h**2 * (rho @ K @ rho.T)                    # V[(pr), (qs)]        return V.reshape(n, n, n, n).transpose(0, 2, 1, 3)    @staticmethod    def antisymmetrize(v):        """<pq|v|rs>_AS = <pq|v|rs> - <pq|v|sr>."""        return v - v.transpose(0, 1, 3, 2)basis = HarmonicOscillatorBasis(n_orb=8)print(f"{basis.n_orb} orbitals on {len(basis.x)} grid points")print(f"|S - I| = {np.linalg.norm(basis.overlap - np.eye(basis.n_orb)):.2e}")print("single-particle energies:", basis.epsilon)

In [ ]:
plt.figure(figsize=(7, 4.2))for n in range(4):    plt.plot(basis.x, basis.phi[n], label=fr"$\phi_{n}$,  $\varepsilon={basis.epsilon[n]}$")plt.xlim(-5, 5)plt.xlabel("$x$")plt.ylabel(r"$\phi_n(x)$")plt.title("The single-particle basis")plt.grid(alpha=0.3)plt.legend()plt.tight_layout()plt.show()

## 2. The Slater determinant$$\Phi(x_1,\dots,x_N)=\frac{1}{\sqrt{N!}}\det\big[\psi_a(x_b)\big]$$Antisymmetry and the Pauli principle are properties of determinants:swapping two particles swaps two *columns* and reverses the sign, whileputting two particles in the same orbital gives two identical *rows* and thedeterminant vanishes.

In [ ]:
def factorial(n):    out = 1    for k in range(2, n + 1):        out *= k    return outclass SlaterDeterminant:    """An antisymmetric N-particle wave function built from orbitals."""    def __init__(self, basis, occupied, coefficients=None):        self.basis = basis        self.occupied = list(occupied)        self.N = len(self.occupied)        if coefficients is None:            C = np.zeros((self.N, basis.n_orb))            for i, p in enumerate(self.occupied):                C[i, p] = 1.0            self.C = C        else:            self.C = np.asarray(coefficients, dtype=float)    def orbitals_on_grid(self):        return self.C @ self.basis.phi    def evaluate(self, coordinates):        """Phi(x_1, ..., x_N) at one set of coordinates."""        psi = self.orbitals_on_grid()        M = np.array([[np.interp(x, self.basis.x, psi[i])                       for x in coordinates] for i in range(self.N)])        return np.linalg.det(M) / np.sqrt(factorial(self.N))    def density_matrix(self):        """rho_{lambda mu} = sum_i C*_{i lambda} C_{i mu}."""        return self.C.T @ self.C    def occupation_numbers(self):        return np.sort(np.linalg.eigvalsh(self.density_matrix()))[::-1]phi = SlaterDeterminant(basis, [0, 1, 2])coords = [0.4, -1.1, 2.0]a = phi.evaluate(coords)b = phi.evaluate([coords[1], coords[0], coords[2]])print(f"Phi(x1,x2,x3) = {a:+.10f}")print(f"Phi(x2,x1,x3) = {b:+.10f}")print(f"sum           = {a + b:+.2e}    <- antisymmetry")print()repeated = SlaterDeterminant(basis, [0, 1, 1])print(f"two particles in orbital 1: Phi = {repeated.evaluate(coords):+.2e}"      f"    <- Pauli principle")

### Occupation numbersThe density matrix $\rho_{\lambda\mu}=\sum_i C^*_{i\lambda}C_{i\mu}$ of aSlater determinant is a **projector**, $\rho^2=\rho$, so its eigenvalues — thenatural occupation numbers of chapter 1 — can only be 0 or 1.  Any deviationfrom those values in a computed state is a direct measure of correlation.

In [ ]:
rho = phi.density_matrix()print("rho^2 = rho ?", np.allclose(rho @ rho, rho))print("occupation numbers:", np.round(phi.occupation_numbers(), 12))

## 3. Basis changesExpanding the orbitals in a fixed basis, $\psi_p=\sum_\lambdaC_{p\lambda}\phi_\lambda$, every entry of the Slater determinant becomes a sum,so by the determinant product rule$$\Phi^{\rm new} = \det(\boldsymbol{C})\,\det(\boldsymbol{\Phi}).$$If $\boldsymbol{C}$ is unitary then $|\det \boldsymbol{C}|=1$ and the state isphysically unchanged.  The consequence is that the **energy is invariant underany unitary rotation among the occupied orbitals** — the freedom exploitedthroughout Hartree-Fock theory.  A rotation that mixes occupied withunoccupied orbitals is a different matter entirely.

In [ ]:
class EnergyFunctional:    """E[Psi] = sum_i <i|h|i> + (1/2) sum_ij <ij|v|ij>_AS, in terms of C."""    def __init__(self, basis):        self.basis = basis        self.h = basis.one_body        self.v = basis.antisymmetrize(basis.two_body())    def energy(self, C):        C = np.asarray(C, dtype=float)        one = np.einsum("ia,ib,ab->", C, C, self.h)        two = np.einsum("ia,jb,ic,jd,abcd->", C, C, C, C, self.v)        return one + 0.5 * two    def energy_from_occupied(self, occupied):        occ = list(occupied)        one = sum(self.h[i, i] for i in occ)        two = sum(self.v[i, j, i, j] for i in occ for j in occ)        return one + 0.5 * two    def fock_matrix(self, C):        """f_ab = h_ab + sum_cd rho_cd <ac|v|bd>_AS."""        rho = C.T @ C        return self.h + np.einsum("cd,acbd->ab", rho, self.v)functional = EnergyFunctional(basis)N = 3C0 = np.eye(basis.n_orb)[:N]E0 = functional.energy(C0)Q, _ = np.linalg.qr(rng.normal(size=(N, N)))     # rotation inside the occupied spaceE1 = functional.energy(Q @ C0)Qfull, _ = np.linalg.qr(rng.normal(size=(basis.n_orb, basis.n_orb)))E2 = functional.energy(Qfull[:, :N].T)           # mixes occupied and virtualprint(f"|det Q|                        = {abs(np.linalg.det(Q)):.10f}")print(f"E, original basis              = {E0:.12f}")print(f"E, rotated occupied space      = {E1:.12f}   (difference "      f"{abs(E1-E0):.1e})")print(f"E, occupied-virtual mixing     = {E2:.12f}   <- this one changes")print()print(f"E from the sum over occupied states = "      f"{functional.energy_from_occupied(range(N)):.12f}")

## 4. The price of the two-body matrix elementsA basis of $n$ single-particle states needs $n^4$ two-body elements.  But thepair-index matrix $V_{(pq),(rs)}$ has a numerical rank far below $n^2$, whichis what the low-rank factorisation of chapter 1 exploits:$$\langle\alpha\beta|v|\gamma\delta\rangle = \sum_{\eta=0}^{M-1} L^\eta_{\alpha\gamma}L^\eta_{\beta\delta},\qquad M \ll n^2 .$$

In [ ]:
n = basis.n_orbpair = basis.two_body().transpose(0, 2, 1, 3).reshape(n*n, n*n)w = np.linalg.eigvalsh(0.5 * (pair + pair.T))[::-1]print(f"n = {n}:  n^4 = {n**4} matrix elements")print(f"pair matrix is {n*n} x {n*n}, numerical rank = "      f"{np.linalg.matrix_rank(pair, tol=1e-10)}")print("leading eigenvalues:", np.array2string(w[:8], precision=4))plt.figure(figsize=(7, 4.2))plt.semilogy(np.maximum(w[:40], 1e-18), "o-", ms=4)plt.xlabel(r"$\eta$")plt.ylabel(r"$\lambda_\eta$")plt.title("Eigenvalues of the two-body matrix in the pair index")plt.grid(alpha=0.3)plt.tight_layout()plt.show()

## 5. Hartree-Fock: a repeated eigenvalue problemThe Hartree-Fock equations say the optimal coefficients are the eigenvectorsof the Fock matrix,$$\sum_\beta f_{\alpha\beta}C_{i\beta}=\varepsilon_i C_{i\alpha},\qquadf_{\alpha\beta}=\langle\alpha|\hat h_0|\beta\rangle  +\sum_{\gamma\delta}\rho_{\gamma\delta}   \langle\alpha\gamma|v|\beta\delta\rangle_{\rm AS},$$but $f$ depends on the solution through $\rho$.  So one guesses, diagonalises,rebuilds and repeats.  This is a preview — Hartree-Fock proper comes later inthe book.

In [ ]:
class HartreeFock:    """A minimal self-consistent field loop."""    def __init__(self, functional, n_particles, max_iter=200, tol=1e-10):        self.functional = functional        self.N = n_particles        self.max_iter = max_iter        self.tol = tol    def solve(self, C0=None):        n = self.functional.basis.n_orb        C = np.eye(n)[:self.N] if C0 is None else np.asarray(C0, float)        energy_old = np.inf        self.history = []        for k in range(self.max_iter):            f = self.functional.fock_matrix(C)            values, vectors = np.linalg.eigh(f)            C = vectors[:, :self.N].T                # lowest N eigenvectors            energy = self.functional.energy(C)            self.history.append(energy)            self.iterations = k + 1            if abs(energy - energy_old) < self.tol:                break            energy_old = energy        self.C, self.epsilon = C, values        return energy, Cprint(f"{'N':>3s} {'E (lowest N orbitals)':>23s} {'E (Hartree-Fock)':>20s} "      f"{'gain':>12s} {'iterations':>12s}")for N in (2, 3, 4):    hf = HartreeFock(functional, N)    E, C = hf.solve()    E_ref = functional.energy_from_occupied(range(N))    print(f"{N:3d} {E_ref:23.8f} {E:20.8f} {E-E_ref:12.2e} "          f"{hf.iterations:12d}")

### What a single determinant missesFor two particles the exact ground state in the same truncated basis can beobtained by diagonalising the Hamiltonian in the space of antisymmetric pairs$|pq\rangle$, $p<q$.  The difference$$E_{\rm corr} = E_{\rm exact} - E_{\rm HF}$$is the correlation energy: what no single Slater determinant can reach, andwhat every method in the rest of the book is built to recover.

In [ ]:
def exact_two_particle(functional):    """Exact energy of two spinless fermions in the truncated basis."""    n = functional.basis.n_orb    pairs = [(p, q) for p in range(n) for q in range(p+1, n)]    H = np.zeros((len(pairs), len(pairs)))    h, v = functional.h, functional.v    for a, (p, q) in enumerate(pairs):        for b, (r, s) in enumerate(pairs):            element = v[p, q, r, s]            if q == s:                element += h[p, r]            if p == r:                element += h[q, s]            if q == r:                element -= h[p, s]            if p == s:                element -= h[q, r]            H[a, b] = element    values, vectors = np.linalg.eigh(H)    return values[0], vectors[:, 0], pairshf = HartreeFock(functional, 2)E_hf, _ = hf.solve()E_exact, psi, pairs = exact_two_particle(functional)print(f"E (Hartree-Fock)      = {E_hf:.10f}")print(f"E (exact, same basis) = {E_exact:.10f}")print(f"correlation energy    = {E_exact - E_hf:+.10f}")print(f"largest amplitude     = {np.max(np.abs(psi)):.6f} on the pair "      f"{pairs[int(np.argmax(np.abs(psi)))]}")

### Correlation grows with the interactionRepeating the comparison as a function of the interaction strength shows thesingle-determinant description degrading in a controlled way: the correlationenergy grows and the largest natural occupation number falls away from one.

In [ ]:
strengths = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]rows = []for g in strengths:    b = HarmonicOscillatorBasis(n_orb=8, strength=g)    f = EnergyFunctional(b)    E_hf, _ = HartreeFock(f, 2).solve()    E_ex, psi, pairs = exact_two_particle(f)    # natural occupations of the exact two-particle state    n = b.n_orb    C = np.zeros((n, n))    for amp, (p, q) in zip(psi, pairs):        C[p, q] += amp / np.sqrt(2)        C[q, p] -= amp / np.sqrt(2)    occ = np.linalg.svd(C, compute_uv=False)**2    # for two fermions the natural occupations come in degenerate pairs, so a    # single Slater determinant has exactly two weights of 1/2 and nothing    # else: the weight in the leading pair is 1 for a determinant and falls    # as correlation sets in    rows.append((g, E_hf, E_ex, E_ex - E_hf, occ[0] + occ[1]))print(f"{'g':>5s} {'E_HF':>13s} {'E_exact':>13s} {'E_corr':>13s} "      f"{'leading weight':>16s}")for g, e1, e2, ec, w0 in rows:    print(f"{g:5.1f} {e1:13.8f} {e2:13.8f} {ec:13.2e} {w0:16.8f}")

In [ ]:
g = [r[0] for r in rows]fig, ax = plt.subplots(1, 2, figsize=(10, 4))ax[0].plot(g, [-r[3] for r in rows], "o-")ax[0].set_xlabel("interaction strength $g$")ax[0].set_ylabel(r"$-E_{\rm corr}$")ax[0].set_title("Correlation energy")ax[0].grid(alpha=0.3)ax[1].plot(g, [r[4] for r in rows], "o-")ax[1].set_xlabel("interaction strength $g$")ax[1].set_ylabel("weight in the leading determinant")ax[1].set_title("Departure from a single determinant")ax[1].grid(alpha=0.3)plt.tight_layout()plt.show()

## Where this leadsChapter 2 has turned the linear algebra of chapter 1 into physics: the Slaterdeterminant is a determinant, a basis change is a unitary transformation, andHartree-Fock is a self-consistent Hermitian eigenvalue problem.What it has also shown is the limit of the construction.  A single Slaterdeterminant has occupation numbers of exactly 0 and 1, so it cannot describecorrelation at all, and the plots above show how quickly that becomes aproblem as the interaction grows.  Second quantisation, developed next,provides the machinery to go beyond one determinant without drowning in thepermutation algebra of section 2.8.